# Certify fault tolerance exactly

Sampling estimates how often a circuit fails under a particular noise model and decoder, with statistical uncertainty that shrinks only as more shots are collected.
Some questions about fault tolerance are structural instead: how many faults does it take to cause an undetected logical error, and how far can a few faults spread?
An exact search can answer these questions for every combination of faults that a noise model allows, independently of their probabilities and of any decoder.
The result either certifies that a circuit meets a fault-tolerance criterion or provides a counterexample that shows where it fails.

This notebook demonstrates two such certificates.
For memory circuits, we compute the fault distance: the minimum number of circuit errors that can flip a logical observable without triggering a detector.
For state preparation circuits, we check bounded fault propagation: whether faults that pass flag post-selection can leave too large an error on the prepared state.
Both searches are exact, but their cost can grow exponentially with the number of faults that they consider.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim
import sympy

from qldpc import circuits, codes

## Fault distance of memory circuits

A syndrome-extraction circuit can spread an ancilla fault to several data qubits, making the gate schedule important to the protection a code provides.
To assess that protection, we compute a circuit's fault distance: the minimum number of error mechanisms that can flip a logical observable without triggering a detector.
We compare a Steane-code memory circuit with a bivariate bicycle memory circuit to see when the extraction circuit preserves the code distance.

### A syndrome extraction circuit that loses distance

The Steane code has distance 3, but the bare-ancilla extraction circuit below has fault distance 2 in each measured logical basis.
Here bare-ancilla extraction measures stabilizers without additional flag ancillas to detect fault propagation.
Initialization and final readout are ideal; only one syndrome-extraction round receives depolarizing noise.
This diagnostic thereby isolates the effect of a noisy syndrome extraction circuit.
A noise model is needed to define which error mechanisms exist, but their probabilities do not affect the results.

In [3]:
noise_model = circuits.DepolarizingNoiseModel(1e-3)


def get_noisy_memory_circuit(code: codes.CSSCode, basis: str, num_rounds: int = 1) -> stim.Circuit:
    """Keep initialization and readout ideal while adding noise to the QEC cycle."""
    parts = circuits.get_memory_experiment_parts(code, basis=basis, num_rounds=num_rounds)
    return parts.initialization + noise_model.noisy_circuit(parts.qec_cycle) + parts.readout


code = codes.SteaneCode()
print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(f"{basis}-basis memory: fault distance {fault_distance}")

code parameters: (7, 1, 3)
X-basis memory: fault distance 2
Z-basis memory: fault distance 2


Distance 2 means that a pair of modeled mechanisms can evade the detectors while flipping a tracked logical observable.
This example shows how a bare-ancilla extraction circuit can have a smaller fault distance than the underlying code.
In this case, distance-preserving syndrome extraction requires flagged, Shor, or Steane-type strategies; see, for example, the [Steane-swap QEC](6_steane_qec.ipynb) notebook.

### What the exact search computes

An error mechanism is an error instruction in the circuit's detector error model (DEM).
Mechanisms with identical detector and observable effects can be merged, so mechanism indices are not physical gate-location indices.
A fault is a set of error mechanisms, and the weight of a fault is the number of error mechanisms it contains.
`circuits.get_fault_distance` searches for a minimum-weight fault with no detector flips and at least one observable flip.
This method converts the problem of finding a minimum-weight fault into that of computing the Z-distance of an auxiliary Calderbank–Shor–Steane (CSS) code, and uses the Brouwer–Zimmermann distance-finding algorithm.
For a derivation of this reduction, see the documentation [here](https://qldpc.readthedocs.io/autoapi/qldpc/circuits/certification/fault_distance/index.html).

### A bivariate bicycle extraction circuit

For comparison, we use a bivariate bicycle code from [arXiv:2308.07915](https://arxiv.org/abs/2308.07915) with parameters $[[72,12,6]]$: 72 physical qubits, 12 logical qubits, and distance 6.
Here each basis has 432 mechanisms after equivalent ones are merged.
Checking all combinations of up to six would require roughly nine trillion combinations; the code-distance reduction avoids that exhaustive enumeration.

In [4]:
x, y = sympy.symbols("x y")
code = codes.BBCode({x: 6, y: 6}, x**3 + y + y**2, y**3 + x + x**2)

print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    num_errors = noisy_circuit.detector_error_model().num_errors
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(f"{basis}-basis memory: {num_errors} error mechanisms, fault distance {fault_distance}")

code parameters: (72, 12, 6)
X-basis memory: 432 error mechanisms, fault distance 6
Z-basis memory: 432 error mechanisms, fault distance 6


The extraction circuit has fault distance 6 in both bases, matching the BB code's distance.
The Steane and BB examples illustrate that distance preservation depends on the code and its extraction circuit.

## Bounded fault propagation in state preparation

For state preparation to be fault-tolerant, a small number of circuit errors should leave only low-weight errors on the prepared state; see [Definition 3 in Section 4.2 of arXiv:0904.2557](https://arxiv.org/abs/0904.2557).
We use the flagged Steane-code preparation circuit from the [state-preparation notebook](4_state_preparation.ipynb), which follows Figure 1c of [Goto (2016)](https://www.nature.com/articles/srep19578).
`certify_bounded_fault_propagation` checks whether any set of circuit errors that passes the flags leaves an error on more data qubits than the set contains.
With `add_flags=True`, it adds a detector for each flag measurement and, by default, post-selects on these detectors.
We can compare results with and without post-selection on flags to see what protection the flags provide.

See the documentation [here](https://qldpc.readthedocs.io/autoapi/qldpc/circuits/certification/fault_propagation/index.html) for additional information about what exactly `certify_bounded_fault_propagation` certifies.
The [certifier guide](https://github.com/qLDPCOrg/qLDPC/blob/main/src/qldpc/circuits/certification/fault_propagation.md) explains why the check is exact and how it searches for violations.

In [5]:
flagged_prep_circuit = stim.Circuit("""
    # non-fault-tolerant state preparation
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")

In [6]:
noisy_prep = noise_model.noisy_circuit(flagged_prep_circuit)

for add_flags in (False, True):
    certificate = circuits.certify_bounded_fault_propagation(noisy_prep, add_flags=add_flags)
    print(f"add_flags={add_flags}: {certificate}")

add_flags=False: Bounded fault propagation violated at fault weight 1 (CSS weight, no post-selection)
  fault of 1 error mechanism:
    X3 Y6 from DEPOLARIZE2 on qubits [3, 6] at instruction 11, tick 3
  output error, up to stabilizers: X on [0, 1], Z on [0, 1, 3]
  residual weight >= 2 > fault weight 1
add_flags=True: Bounded fault propagation certified up to any fault weight (CSS weight, post-selecting on 1 detector)


Without post-selection, the check finds a single fault that leaves too large an error on the prepared state.
With flag post-selection, the same circuit passes the bounded-propagation check.

Printing a certificate states its result and the settings that it covers.
For a counterexample, the printout traces each error mechanism of the fault to one circuit error that produces it: the Pauli error, the noise instruction and qubits where it occurs, and the instruction's index (from 0) and tick in the noisy circuit.
It also shows the qubit supports of an error that is equivalent, up to stabilizers of the prepared state, to the error that the fault leaves.
The printed residual weight is a lower bound that exceeds the fault weight, which suffices to show a violation; the exact residual weight may be larger.

To inspect a counterexample further, `certificate.diagnostic_circuit` is the circuit that the search analyzed: the noisy circuit, followed by the flag detectors and a noiseless detector for each stabilizer of the prepared state.
Its detector error model is `certificate.detector_error_model`, whose error instructions the counterexample's `fault_indices` refer to, so Stim's `explain_detector_error_model_errors` can locate any fault in the circuit.

The [state-preparation notebook](4_state_preparation.ipynb) samples the logical error and discard rates of this flagged circuit, and the [Steane-swap QEC notebook](6_steane_qec.ipynb) uses it within a complete error-correction cycle.